In [0]:
##### Dashboard 1 -- single page - so can view in notebook
# NOTE: UPDATED TO ONLY SHOW LAST 20 RUNS to limit data and not exceed 20mb limit on notebooks
# dbutils.library.restartPython()
import sys
from pathlib import Path
# Go UP from Reports → root, then add root to path
sys.path.append(str(Path().resolve().parent))
from functions import spa_dashboard
import importlib

importlib.reload(spa_dashboard)

# Current Databases
# runs_table = "test_automation_runs"
# results_table = "test_automation_results"

runs_table = "test_automation_runs2"
results_table = "test_automation_results2"

html_path, html_str = spa_dashboard.generate_dashboard_single_page(runs_table, results_table)

displayHTML(html_str)

In [0]:
#####Dashbord 2 -- Separate files
#Use to download and view locally
#databricks fs cp --recursive dbfs:/tmp/ ./
import functions.multi_file_dashboard as dashboard
import importlib
importlib.reload(dashboard)

dashboard.generate_dashboard()

In [0]:
from databricks.sdk.runtime import *
from pyspark.sql import functions as F

run_id = "3206e491-f384-48e2-bca6-2e4c25ca015a"
results_table = "test_automation_results2"

df = (
    spark.table(results_table)
    .filter(F.col("run_id") == run_id)
    .withColumn("status_norm", F.upper(F.trim(F.coalesce(F.col("status"), F.lit("")))))
)

display(
    df.groupBy("status", "status_norm")
      .count()
      .orderBy("status_norm", "status")
)

totals = df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("status_norm").isin("PASS", "PASSED"), 1).otherwise(0)).alias("pass_count"),
    F.sum(F.when(F.col("status_norm").isin("FAIL", "FAILED"), 1).otherwise(0)).alias("fail_count"),
    F.sum(F.when(F.col("status_norm") == "ERROR", 1).otherwise(0)).alias("error_count"),
    F.sum(F.when(F.col("status_norm").isin("NO_DATA", "NO DATA", "NODATA"), 1).otherwise(0)).alias("no_data_count"),
).collect()[0]

print({
    "total_rows": totals["total_rows"],
    "pass_count": totals["pass_count"],
    "fail_count": totals["fail_count"],
    "error_count": totals["error_count"],
    "no_data_count": totals["no_data_count"],
    "bucket_sum": totals["pass_count"] + totals["fail_count"] + totals["error_count"] + totals["no_data_count"],
})

In [0]:
# ===== PARAMETERS =====
runs_table = "test_automation_runs2"
results_table = "test_automation_results2"
show_last = 20
# ======================

from pyspark.sql.functions import col, desc
import pandas as pd

runs_df = spark.table(runs_table)
runs = runs_df.filter(col("run_by_automation_name") == "Transformation_Tests") \
    .orderBy(desc("run_start_datetime")).limit(show_last).toPandas()

results_df = spark.table(results_table)

# Count results per run broken down by status
run_stats = {}
for run_id in runs["run_id"]:
    rdf = results_df.filter(col("run_id") == run_id).toPandas()
    statuses = rdf["status"].str.upper().str.strip() if not rdf.empty else pd.Series(dtype=str)
    run_stats[run_id] = {
        "total": len(rdf),
        "pass": (statuses == "PASS").sum(),
        "fail": (statuses == "FAIL").sum(),
        "nodata": (statuses == "NO_DATA").sum(),
        "error": (statuses == "ERROR").sum(),
    }

rows_html = ""
for _, r in runs.iterrows():
    rid = r["run_id"]
    s = run_stats.get(rid, {"total": 0, "pass": 0, "fail": 0, "nodata": 0, "error": 0})
    status_css = "pass" if r.get("run_status") == "PASS" else "fail"
    rows_html += f'''<tr>
        <td><input type="text" value="{rid}" readonly
             onclick="this.select()"
             style="font-family:monospace;font-size:11px;width:290px;border:1px solid #ccc;padding:2px 4px;cursor:pointer"
             title="Click to select, then Ctrl+C"></td>
        <td>{r.get("state_under_test","")}</td>
        <td>{str(r.get("run_start_datetime",""))[:16]}</td>
        <td class="{status_css}">{r.get("run_status","")}</td>
        <td class="pass">{s["pass"]}</td>
        <td class="fail">{s["fail"]}</td>
        <td class="nodata">{s["nodata"]}</td>
        <td class="error">{s["error"]}</td>
        <td>{s["total"]}</td>
        <td>{r.get("run_user","")}</td>
    </tr>\n'''

html = f"""
<style>
    .mgr {{font-family:Arial,sans-serif;padding:10px}}
    .mgr table {{border-collapse:collapse;width:100%;font-size:13px}}
    .mgr th {{background:#2c3e50;color:white;padding:6px 10px;text-align:left}}
    .mgr td {{border:1px solid #ddd;padding:4px 8px}}
    .mgr tr:nth-child(even) {{background:#f9f9f9}}
    .mgr .pass {{color:#27ae60;font-weight:bold}}
    .mgr .fail {{color:#e74c3c;font-weight:bold}}
    .mgr .nodata {{color:#f39c12;font-weight:bold}}
    .mgr .error {{color:#7f8c8d;font-weight:bold}}
    .mgr .info {{background:#ecf0f1;padding:12px;border-left:4px solid #2c3e50;margin:12px 0}}
</style>
<div class="mgr">
    <h2>Test Run Manager</h2>
    <p>Tables: <code>{runs_table}</code> / <code>{results_table}</code> | Last {show_last} runs</p>
    <div class="info">
        <b>To delete a run:</b> Click the run ID field (auto-selects), Ctrl+C to copy, paste into Cell 2 below, run Cell 2.
    </div>
    <table>
        <thead><tr><th>Run ID (click to select)</th><th>State</th><th>Date</th><th>Status</th><th>Pass</th><th>Fail</th><th>No Data</th><th>Error</th><th>Total</th><th>User</th></tr></thead>
        <tbody>{rows_html}</tbody>
    </table>
</div>
"""
displayHTML(html)

In [0]:
# ===== PASTE RUN ID HERE =====
delete_run_id = "f6285679-6001-4d30-b196-c4e2f7858c98"   # e.g. "3f2e82ae-39ee-42bb-b0eb-28f0a168aa1b"

# Tables (same as Cell 1)
runs_table = "test_automation_runs2"
results_table = "test_automation_results2"
# ==============================

if not delete_run_id:
    print("Set delete_run_id above and re-run this cell")
else:
    from pyspark.sql.functions import col

    run = spark.table(runs_table).filter(col("run_id") == delete_run_id).first()
    if not run:
        print(f"Run not found: {delete_run_id}")
    else:
        state = run.state_under_test
        date = str(run.run_start_datetime)[:16]
        result_count = spark.table(results_table).filter(col("run_id") == delete_run_id).count()

        print(f"Deleting run: {delete_run_id[:8]}... | {state} | {date} | {result_count} results")

        spark.sql(f"DELETE FROM {results_table} WHERE run_id = '{delete_run_id}'")
        print(f"  Deleted {result_count} results from {results_table}")

        spark.sql(f"DELETE FROM {runs_table} WHERE run_id = '{delete_run_id}'")
        print(f"  Deleted run from {runs_table}")

        print(f"\nDone. Run {delete_run_id[:8]}... removed.")

    delete_run_id = ""

In [0]:
######WIP - Dashboard3 - files on FileStore - try and use html in notebook / get via http files/...avoid 20mb memory limit on single file

# index_dbfs_path = dashboard.generate_dashboard_scalable()
# index_dbfs_path, index_web_path = dashboard.generate_dashboard_scalable()
# print("DBFS file:", index_dbfs_path)
# print("Open in browser:", index_web_path)

# from IPython.display import HTML, display
# display(HTML(f'<a href="{index_web_path}" target="_blank">Open ARIA Dashboard</a>'))

# from IPython.display import HTML, display
# display(HTML(f"""
# <iframe src="{index_web_path}" style="width:100%; height:900px; border:1px solid #ddd; border-radius:8px;"></iframe>
# """))

# with open(index_dbfs_path, "r", encoding="utf-8") as f:
#     html = f.read()

# displayHTML(html)

# from functions.dashboard import generate_dashboard_option1
import functions.filestore_dashboard as dashboard
import importlib
importlib.reload(dashboard)

index_dbfs_path = dashboard.generate_dashboard_option1()

print(index_dbfs_path)
displayHTML(f'<a href="/files/aria_dashboard/index.html" target="_blank">Open ARIA Dashboard</a>')


with open(index_dbfs_path, "r", encoding="utf-8") as f:
    html = f.read()
displayHTML(html)